# P8-4 · Backend và dữ liệu cho AI

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import hashlib, json, fakeredis
r = fakeredis.FakeRedis()          # chạy thử không cần server; thật: import redis; r = redis.Redis(host="localhost", port=6379)

calls = 0
def model_predict(text):            # thay cho model thật, đếm số lần bị gọi
    global calls; calls += 1
    return {"label": "tích cực", "len": len(text)}

def cached_predict(text, ttl=3600):
    key = "pred:v3:" + hashlib.sha256(text.encode()).hexdigest()   # v3: phiên bản model/prompt
    if (hit := r.get(key)) is not None:
        return json.loads(hit)
    out = model_predict(text)
    r.setex(key, ttl, json.dumps(out))
    return out

for q in ["ghế đẹp", "ghế đẹp", "giao chậm", "ghế đẹp"]:
    cached_predict(q)
print("4 yêu cầu, model chỉ chạy", calls, "lần; TTL còn lại:", r.ttl("pred:v3:" + hashlib.sha256("ghế đẹp".encode()).hexdigest()), "giây")

4 yêu cầu, model chỉ chạy 2 lần; TTL còn lại: 3599 giây


## Bài tập 5: Truy vấn cho bộ eval {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần Postgres có bảng conversations, messages của hệ thống bạn
import json, psycopg

SQL = """
WITH ranked AS (
  SELECT c.customer_id, m.content AS question, length(m.content) AS len,
         ROW_NUMBER() OVER (PARTITION BY c.customer_id ORDER BY length(m.content) DESC) AS rn
  FROM conversations c
  JOIN messages m ON m.conversation_id = c.id AND m.role = 'user'
  WHERE c.created_at >= date_trunc('month', now()) - interval '1 month'
    AND c.created_at <  date_trunc('month', now())
    AND (c.feedback = 'dislike' OR c.escalated)
)
SELECT customer_id, question FROM ranked WHERE rn <= 2 ORDER BY len DESC LIMIT 200;
"""

def export_eval(dsn, path="eval_candidates.jsonl"):
    with psycopg.connect(dsn) as conn, open(path, "w", encoding="utf-8") as f:
        for customer_id, question in conn.execute(SQL):
            f.write(json.dumps({"customer_id": customer_id, "question": question}, ensure_ascii=False) + "\n")